In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

PROBLEM 1: Data analysis using markov chians 

In this problem, you will empirically analyze a Markov chain 
with a finite state space. Transition probabilities are unknown.

The state space is:
    S = {0, 1, 2, 3}

You are given the data for the observed X_t for t  = 0..19

Tasks:
1. Estimate the transition matrix P from the observed transitions.
2. Verify that the estimated matrix is a probability transition matrix.
3. Compute the stationary distribution pi of the chain.
4. Simulate the chain using the estimated transition matrix
5. Compute the expected hitting times via

   (a) Simulation

   (b) Solving linear equations (analytical hitting times). 

Compare the estimates and interpret the results


In [2]:
import numpy as np

# state space
S = [0, 1, 2, 3]
N_states = len(S)

# Observed transitions: each row is (current_state, next_state)
X_t = np.array([
    [0, 1],
    [1, 2],
    [2, 3],
    [3, 0],
    [0, 1],
    [1, 1],
    [1, 2],
    [2, 2],
    [2, 3],
    [3, 3],
    [3, 0],
    [0, 2],
    [2, 1],
    [1, 3],
    [3, 1],
    [1, 0],
    [0, 0],
    [0, 1],
    [1, 2],
    [2, 0],
], dtype=int)




Below are methods that you need to complete

In [ ]:
# 1.1 Estimate the transition matrix P from the observed transitions.
def comp_transition_matrix(transitions, n_states):
    """
    Estimate the transition matrix P from observed transitions.

    Args:
        transitions: array of shape (n_samples, 2)
        n_states: number of states

    Returns:
        P_hat: estimated transition matrix
    """
    P_hat = np.zeros((n_states, n_states))

    # Count transitions
    for current_state, next_state in transitions:
        P_hat[current_state, next_state] += 1

    # Normalize rows to get probabilities
    for i in range(n_states):
        row_sum = P_hat[i].sum()
        if row_sum > 0:  # avoid division by zero
            P_hat[i] /= row_sum

    return P_hat

#  1.2 Verify that the estimated matrix is a probability transition matrix.
def is_transition_matrix(P):
    """
    Check if P is a transition matrix.
    """
    if P.shape[0] != P.shape[1]:
        return False

    # all entries must be non-negative
    if np.any(P < 0):
        return False

    # each row must sum to 1
    for i in range(P.shape[0]):
        if not np.isclose(P[i].sum(), 1):
            return False

    return True


# 1.3  Compute the stationary distribution pi of the chain.

def stationary_distribution(P):
    """
    Compute stationary distribution
    """
    
    eigenvalues, eigenvectors = np.linalg.eig(P.T)

    # Find index of eigenvalue close to 1
    idx = np.argmin(np.abs(eigenvalues - 1))

    # Corresponding eigenvector
    pi = np.real(eigenvectors[:, idx])

    # Normalize to make it a probability distribution
    pi = pi / pi.sum()

    return pi

#1.4 Simulate the chain using the estimated transition matrix
def simulate_chain(P, start_state, n_steps):
    """
    Simulate a Markov chain trajectory with a fixed random seed.

    Returns: array of visited states of length n_steps + 1
    """
    seed = 1234 # don't change that
    
    rng = np.random.default_rng(seed)


    path = np.zeros(n_steps + 1, dtype=int)
    path[0] = start_state
    for t in range(n_steps):
            current_state = path[t]
            path[t + 1] = rng.choice(
            P.shape[0],          # states: 0, 1, ..., n-1
            p=P[current_state]   # transition probabilities from current_state
        )

    return path

    #  sample next states using rng.choice

#Compute the expected hitting times via

#   (a) Simulation
#  (b) Solving linear equations (analytical hitting times).

 
def hitting_times_sim(P, start_state, n_sim=10_000):
    """
    Estimate expected hitting times E[T_{start -> j}] for ALL states j.

    Returns:
        est: 1D array, where est[j] the estimated expected steps to hit state j from start_state. 
    """
    
    est = np.full(N_states, np.nan, dtype=float)
    seed = 1234
    rng = np.random.default_rng(seed)

    max_steps = 10000  # safety cap to prevent infinite loops if a state is unreachable

    for target in range(N_states):
        # If we start at the target, hitting time is 0 by definition
        if target == start_state:
            est[target] = 1.0
            continue

        times = np.empty(n_sim, dtype=float)

        for sim in range(n_sim):
            state = start_state
            t = 0

            # Simulate until we hit 'target' or reach max_steps
            while state != target and t < max_steps:
                state = rng.choice(P.shape[0], p=P[state])
                t += 1

            # If we hit the target, store t; otherwise store NaN (ignored in mean)
            times[sim] = t if state == target else np.nan

        est[target] = np.nanmean(times)

    return est


#Compare the estimates and interpret the results

def theoretical_hitting_times(P, start_state):

    n = P.shape[0]
    hit_theor = np.full(n, np.nan, dtype=float)

    for target in range(n):
        # If start == target, hitting time is 0
        if target == start_state:
            hit_theor[target] = 1.0
            continue

        # Build linear system Ah = b
        A = np.eye(n)
        b = np.ones(n)

        # Equation for target: h_target = 0
        A[target, :] = 0
        A[target, target] = 1
        b[target] = 0

        # Equations for i != target:
        # h_i - sum_k P_{ik} h_k = 1
        for i in range(n):
            if i != target:
                A[i, :] -= P[i]

        # Solve system
        h = np.linalg.solve(A, b)

        # Store hitting time from start_state
        hit_theor[target] = h[start_state]

    return hit_theor

When you are done, run the following cell (no need to implement anything else)

In [4]:
def problem1_main():
    print("\n=== Problem 1: Markov chain estimation + hitting times ===")

    # 1) Estimate P
    P_hat = comp_transition_matrix(X_t, N_states)
    print("Estimated P_hat:\n", np.round(P_hat, 3))

    # 2) Validate
    print("Is valid transition matrix?", is_transition_matrix(P_hat))

    # 3) Expected steps from given start state to all states
    start_state = 0

    # simulation
    mc = hitting_times_sim(P_hat, start_state=start_state, n_sim=5000)

    # Theory (linear system)
    th = theoretical_hitting_times(P_hat, start_state=start_state)

    # 4) Compare
    df = pd.DataFrame({
        "target_state": np.arange(N_states),
        "MC_estimate": mc,
        "theoretical": th,
        "abs_diff": np.abs(mc - th),
    })
    print("\nComparison table:\n", df)

In [5]:
problem1_main()


=== Problem 1: Markov chain estimation + hitting times ===
Estimated P_hat:
 [[0.2   0.6   0.2   0.   ]
 [0.167 0.167 0.5   0.167]
 [0.2   0.2   0.2   0.4  ]
 [0.5   0.25  0.    0.25 ]]
Is valid transition matrix? True

Comparison table:
    target_state  MC_estimate  theoretical  abs_diff
0             0       1.0000     1.000000  0.000000
1             1       2.0154     2.024390  0.008990
2             2       3.2912     3.317073  0.025873
3             3       5.6374     5.682927  0.045527


PROBLEM 2: Cost-Sensitive Classification

You are given a binary classification problem for fraud detection.

Class labels:

    y = 1 => fraud

    y = 0 => ok



The costs of classification outcomes are:
    TP = 0, TN = 0, FP = 100, FN = 500

Tasks:
1. Train an SVM classifier.
2. Compute classification costs at a fixed threshold (0.5).
3. Evaluate total cost for multiple probability thresholds.
4. Find the threshold that minimizes total cost.

In [6]:
import numpy as np
import pandas as pd

costs = {"TP": 0, "TN": 0, "FP": 100, "FN": 500}


def generate_fraud_table(seed=0, n=3000, fraud_rate=0.05):
    """
    Generate a simple fraud dataset as a single table. The table contains:
        - numerical features: x1, x2, x3
        - binary target column: fraud (1 = fraud, 0 = legitimate)
    """
    rng = np.random.default_rng(seed)

    # Target variable
    fraud = (rng.random(n) < fraud_rate).astype(int)

    # Features
    x1 = rng.normal(0, 1, size=n)
    x2 = rng.normal(0, 1, size=n)
    x3 = rng.normal(0, 1, size=n)

    #  fraud cases are shifted
    x1[fraud == 1] += 2.0
    x2[fraud == 1] += 1.0

    df = pd.DataFrame({
        "x1": x1,
        "x2": x2,
        "x3": x3,
        "fraud": fraud,
    })

    return df


fraud_data = generate_fraud_table()

fraud_data.head()

,x1,x2,x3,fraud
0,-0.250243,-0.863902,-0.307019,0
1,-0.380736,0.018756,-0.559577,0
2,1.126431,2.055912,0.973126,1
3,0.806991,2.104160,-0.211368,1
4,0.059649,0.652374,-0.437259,0


Fill in the methods in the cell below:

In [7]:
#from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split


def train_test_split_table(df):
    """
    Split a data table into training and test sets.

    Returns:
        X_train, X_test, y_train, y_test
    """
    # implement splitting
    # first, decide what are features and what are target 
    X = df[["x1", "x2", "x3"]]   # features
    y = df["fraud"]              # target

    # 2. Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=0,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

def fit_linear_svm(fraud_data):
    """
    Fit a linear SVM classifier.

    Args: data table

    Returns:
        predicted labels of length len(y_test) 
    """
    # define our model
    clf = LinearSVC(
        C=1.0,
        max_iter=10_000,
        random_state=0
    )

    # split the data into train and test
    X_train, X_test, y_train, y_test = train_test_split_table(fraud_data)

    # fit the SVM
    clf.fit(X_train, y_train)

    # predict labels on the test set
    y_pred = clf.predict(X_test)

    return clf


def confusion_counts(y_true, y_pred):
    
    """
    Computes TP, TN, FP, FN.
    """
    
    TP_est, TN_est, FP_est, FN_est = 0, 0, 0, 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 1 and yp == 1:
            TP_est += 1
        elif yt == 0 and yp == 0:
            TN_est += 1
        elif yt == 0 and yp == 1:
            FP_est += 1
        elif yt == 1 and yp == 0:
            FN_est += 1

    return {"TP": TP_est, "TN": TN_est, "FP": FP_est, "FN": FN_est}


def total_cost(counts):
    """
    Compute total cost from confusion counts.

    """
    # Multiply counts by costs and sum
    total_cost = (
        costs["TP"] * counts["TP"]
        + costs["TN"] * counts["TN"]
        + costs["FP"] * counts["FP"]
        + costs["FN"] * counts["FN"]
    )
    
    return total_cost


# evaluate how the classification cost changes when you change the decision threshold.
def sweep_thresholds(y_true, clf, X, thresholds):
    """
    Evaluate total cost for a range of thresholds.
    
    Here, clf is your trained SVM classifier
    """

    results = []
    
    # note: here, I define y_probs to be just a decision function. Think: does it need to be calibrated to be used in this problem?
    y_probs = clf.decision_function(X)

    for t in thresholds:
        # 1) compute the prediction for a chosen theshold
        y_pred = (y_probs >= t).astype(int)

        # 2) Confusion matrix counts  (previoulsy implemented by you)
        counts = confusion_counts(y_true, y_pred)

        # 3) Total cost (previoulsly implemented by you)
        cost = total_cost(counts)

        # 4) Store results
        results.append({
            "threshold": t,
            "TP": counts["TP"],
            "TN": counts["TN"],
            "FP": counts["FP"],
            "FN": counts["FN"],
            "total_cost": cost,
        })

    return pd.DataFrame(results)



When you are done, run the following cell (no need to implement anything else)

In [8]:
def main():

    df = fraud_data

    print("Dataset head:")
    print(df.head(), "\n")

    # split in train and test:
    _, X_test, _, y_test = train_test_split_table(df)
    # Fit linear SVM
    clf = fit_linear_svm(df)

    # thresholds
    thresholds = np.linspace(-2.0, 2.0, 21)
    df_results = sweep_thresholds(
        y_test,
        clf,
        X_test,
        thresholds,
    )

    print("Threshold sweep results:")
    print(df_results)

    # 6) Identify optimal threshold
    best_row = df_results.loc[df_results["total_cost"].idxmin()]
    print("Optimal threshold:", best_row)

In [9]:
main()

Dataset head:
         x1        x2        x3  fraud
0 -0.250243 -0.863902 -0.307019      0
1 -0.380736  0.018756 -0.559577      0
2  1.126431  2.055912  0.973126      1
3  0.806991  2.104160 -0.211368      1
4  0.059649  0.652374 -0.437259      0 

Threshold sweep results:
    threshold  TP   TN   FP  FN  total_cost
0        -2.0  48  233  619   0       61900
1        -1.8  48  337  515   0       51500
2        -1.6  48  428  424   0       42400
3        -1.4  48  533  319   0       31900
4        -1.2  48  639  213   0       21300
5        -1.0  47  714  138   1       14300
6        -0.8  47  764   88   1        9300
7        -0.6  43  800   52   5        7700
8        -0.4  38  817   35  10        8500
9        -0.2  25  841   11  23       12600
10        0.0  18  850    2  30       15200
11        0.2  15  852    0  33       16500
12        0.4   8  852    0  40       20000
13        0.6   4  852    0  44       22000
14        0.8   2  852    0  46       23000
15        1.0   1  85

PROBLEM 3: Confidence estimation of the cost

In Problem 2, you trained a classifier, selected a decision threshold, evaluated its performance on a test set, and computed the cost

In this problem, you will quantify the uncertainty of this estimated cost. Each observation in the test set produces a cost depending on the
classification outcome:

    TN: 0
   
    FP: 100

    TP: 0

    FN: 500

Thus, the cost per observation is a bounded random variable taking
values in the interval [0, 500].

Tasks:
1. Compute the average cost per observation on the test set.
2. Use Hoeffding’s inequality to construct a 95% confidence interval
   for the true expected cost of the classifier.
3. Interpret the resulting interval:
   - What does it say about the reliability of your estimate?
   - Is the interval likely to be tight or conservative? Why?

You may assume that test observations are independent and identically
distributed.

In [10]:
def per_observation_cost(y_true, y_pred):
    """
    Compute per-observation cost vector.
    """
    costs_vec = np.zeros_like(y_true, dtype=float)

    for i, (yt, yp) in enumerate(zip(y_true, y_pred)):
        if yt == 0 and yp == 1:        # False Positive
            costs_vec[i] = 100
        elif yt == 1 and yp == 0:      # False Negative
            costs_vec[i] = 500
        else:                          # TP or TN
            costs_vec[i] = 0

    # average cost per observation
    cost_avg = costs_vec.mean()

    return costs_vec, cost_avg

def hoeffding_ci(per_obs_costs, mean, n, a, b, delta=0.05):
    """
    Hoeffding confidence interval
    """
    # Step 1: deterministic costs per observation
    c = per_obs_costs

    # Step 2: average cost
    mean_cost = np.mean(c)

    # Step 3: Hoeffding radius
    epsilon = (b - a) * np.sqrt(np.log(2 / delta) / (2 * n))

    # Confidence interval
    ci = (mean_cost - epsilon, mean_cost + epsilon)

    return ci

The Hoeffding confidence interval guarantees that the true expected cost lies within the interval with high probability, providing a reliable but conservative uncertainty estimate. Because it relies only on boundedness and not on the empirical variance, the resulting interval is typically wider than necessary in practice